# From Trajectory Probability to Policy Gradient

Derive the score-function estimator and verify it on an enumerable environment.

## Before You Start

Course 12. Entry course: 10. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** rl-01_policy.

## What You Will Be Able to Do

- Compute log trajectory probabilities
- Use the log-derivative identity
- Explain why model-free gradients need no transition derivative

## Multiply probabilities, add log probabilities

A trajectory probability factors into an initial-state probability, policy action probabilities, and environment transitions. For a fixed environment whose dynamics do not depend on policy parameters, only the policy factors contribute to the parameter derivative of log probability. Thus grad log P(trajectory)=sum_t grad log pi(a_t|s_t). This is why a policy network can be trained model-free: it need not learn or differentiate an environment-transition model.

For Bernoulli sigmoid policies, the chosen-action log probability is a*log(p)+(1-a)*log(1-p), and its logit derivative is a-p. A length-100 trajectory with action probability .5 at each step has probability 2^-100, about 7.9e-31. Sum stable log probabilities instead of multiplying many small numbers. The environment probability factors cancel from the score derivative, not from the actual distribution of sampled trajectories. In code, select z for action 1 or -z for action 0 and evaluate -logaddexp(0,-selected_z), avoiding zero times an unchosen infinite log probability. Compute both sigmoid tails independently and select score q for action 1 or -p for action 0. Promote integer inputs before arithmetic; sum scores over time. NaN computed logits or nonfinite accumulated outputs raise FloatingPointError, and float64 tail underflow can precede feature scaling.

## The expectation supplies the learning signal

Let J(theta)=sum_tau P_theta(tau) R(tau). Differentiate: grad J=sum_tau R(tau) grad P_theta(tau)=sum_tau P_theta(tau) R(tau) grad log P_theta(tau). Sampling on-policy trajectories therefore estimates the gradient with return times the summed score. Treat sampled states/actions/rewards as fixed during this estimator; do not backpropagate through a discrete action sample.

A reward obtained at the end supplies weak information about many earlier decisions. The estimator can have high variance, especially with long trajectories or rare successes. A state-dependent baseline independent of the sampled action can reduce variance without changing the expectation under appropriate sampling. The lab and experiment enumerate a two-action sequence, compute the exact expected-return derivative, and compare Monte Carlo and finite-difference estimates.

## Worked Example: Two actions

Action probabilities are .8 and .25 for the actions actually taken; final reward is 3.

1. **Probability:** Conditional action product=.8*.25=.2. Environmental factors are omitted only in this fixed-transition toy.
2. **Log probability:** log(.8)+log(.25)=log(.2), about -1.609.
3. **Gradient contribution:** Multiply the sum of chosen-action score gradients by 3. Do not multiply the score vectors together.

The product rule becomes a sum of score terms after taking a logarithm.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Stable Bernoulli trajectory log probability and score gradient."""
import numpy as np

def trajectory_score(states, actions, weights, bias):
    """Return (log action-probability product, grad_w, grad_b).

states [T,D], actions binary [T], weights [D], finite bias. Valid inputs
assumed; T may be zero. Environment factors are not included. Sum scores
over time. Compute in floating point, including for integer inputs.
Stable for extreme logits, including signed infinite logits from overflow
when the selected actions have representable log probabilities and scores.
Raise FloatingPointError for NaN logits or nonfinite computed log
probability or gradients in float64. Do not mutate inputs.
Evaluate both sigmoid tails independently in float64. Tails below its
subnormal range underflow to zero before later score-gradient scaling,
even when an exact tail times a large state would be representable."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

Four chosen actions each have probability .5.

- Action probability product

### Worked Answers

- 0.0625: (.5)^4=1/16.

## Experiment

Run `experiments/rl_gradients.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- The exact enumerable objective should agree with finite differences; Monte Carlo estimates vary with samples and seeds.
- A neural policy is still model-free when it does not model environment transitions.

## Transfer Problems

1. Add a constant baseline and verify unchanged exact expectation.
2. Construct a sparse terminal reward and measure estimator variance as horizon grows.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[Sutton and Barto: Policy Gradient Methods](https://web.stanford.edu/class/psych209/Readings/SuttonBartoRL.pdf). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.


## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Stable Bernoulli trajectory log probability and score gradient."""
import numpy as np

def trajectory_score(states, actions, weights, bias):
    """Return (log action-probability product, grad_w, grad_b).

states [T,D], actions binary [T], weights [D], finite bias. Valid inputs
assumed; T may be zero. Environment factors are not included. Sum scores
over time. Compute in floating point, including for integer inputs.
Stable for extreme logits, including signed infinite logits from overflow
when the selected actions have representable log probabilities and scores.
Raise FloatingPointError for NaN logits or nonfinite computed log
probability or gradients in float64. Do not mutate inputs.
Evaluate both sigmoid tails independently in float64. Tails below its
subnormal range underflow to zero before later score-gradient scaling,
even when an exact tail times a large state would be representable."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
x = np.array([[1., 2.], [-1., 3.], [2., 0.]])
a, w, b = np.array([1, 0, 1]), np.array([.2, -.4]), .1
lp, gw, gb = trajectory_score(x, a, w, b)
p = 1/(1+np.exp(-(x@w+b)))
assert np.isclose(np.exp(lp), np.prod(np.where(a, p, 1-p)))
eps = 1e-5
for i in range(2):
    d = np.eye(2)[i]*eps
    assert np.isclose(gw[i], (trajectory_score(x,a,w+d,b)[0]-trajectory_score(x,a,w-d,b)[0])/(2*eps))
assert np.isclose(gb, (trajectory_score(x,a,w,b+eps)[0]-trajectory_score(x,a,w,b-eps)[0])/(2*eps))
for states, actions, weights, bias in [
    (np.array([[200]], dtype=np.uint8), np.array([1], dtype=np.uint8), np.array([2], dtype=np.uint8), np.int16(-400)),
    (np.array([[100, 100]], dtype=np.int8), np.array([0], dtype=np.uint8), np.array([2, -1], dtype=np.int8), np.int16(-100)),
    (np.array([[-100]], dtype=np.int8), np.array([1], dtype=np.int8), np.array([2], dtype=np.int8), np.int16(200)),
]:
    before = [value.copy() for value in (states, actions, weights)]
    log_probability, gradient, bias_gradient = trajectory_score(states, actions, weights, bias)
    score = float(actions[0]) - .5
    assert np.isclose(log_probability, -np.log(2.))
    assert np.array_equal(gradient, states.astype(float)[0] * score) and bias_gradient == score, 'Promote integer operands before matmul and signed cancellation'
    assert gradient.dtype.kind == 'f' and isinstance(log_probability, float) and isinstance(bias_gradient, float)
    assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
assert np.isfinite(trajectory_score(np.array([[1000.]]), np.array([0]), np.ones(1), 0)[0])
for state, action in [(1e308, 1), (-1e308, 0)]:
    states, actions, weights = np.array([[state]]), np.array([action]), np.array([2.])
    before = [value.copy() for value in (states, actions, weights)]
    log_probability, gradient, bias_gradient = trajectory_score(states, actions, weights, 0.)
    assert log_probability == 0. and np.array_equal(gradient, [0.]) and bias_gradient == 0., 'Saturated matching actions must not multiply an unchosen infinite log probability by zero'
    assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
for state, action in [(1e308, 0), (-1e308, 1)]:
    log_probability, gradient, bias_gradient = trajectory_score(np.array([[state]]), np.array([action]), np.ones(1), 0.)
    assert log_probability == -1e308 and np.array_equal(gradient, [-1e308])
    assert bias_gradient == (1. if action else -1.), 'Finite extreme logits retain representable opposing-action scores'
for states, actions, weights, bias in [
    (np.array([[1e308]]), np.array([0]), np.array([2.]), 0.),
    (np.array([[-1e308]]), np.array([1]), np.array([2.]), 0.),
    (np.zeros((2, 1)), np.ones(2), np.zeros(1), -1e308),
    (np.full((4, 1), 1e308), np.ones(4), np.zeros(1), 0.),
    (np.array([[0.]]), np.ones(1), np.array([np.inf]), 0.),
]:
    before = [value.copy() for value in (states, actions, weights)]
    try:
        trajectory_score(states, actions, weights, bias)
    except FloatingPointError:
        pass
    else:
        raise AssertionError('Nonrepresentable trajectory outputs and NaN logits must fail explicitly')
    assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
empty = trajectory_score(np.empty((0,2)), np.array([]), w, b)
assert empty[0] == 0 and np.allclose(empty[1], 0) and empty[2] == 0
tail = np.exp(-40.)/(1+np.exp(-40.))
for sign in [1., -1.]:
    for action in [0, 1]:
        states, actions, weights = np.array([[1e18]]), np.array([action]), np.array([sign*4e-17])
        before = [value.copy() for value in (states, actions, weights)]
        log_probability, gradient, bias_gradient = trajectory_score(states, actions, weights, 0.)
        matching = action == int(sign > 0)
        expected_score = (1. if action else -1.)*(tail if matching else 1-tail)
        expected_log_probability = -np.log1p(np.exp(-40.)) if matching else -40.-np.log1p(np.exp(-40.))
        np.testing.assert_allclose(log_probability, expected_log_probability, rtol=1e-14, atol=0.)
        np.testing.assert_allclose(gradient, [1e18*expected_score], rtol=1e-14, atol=0.)
        np.testing.assert_allclose(bias_gradient, expected_score, rtol=1e-14, atol=0.)
        assert log_probability < 0 and gradient[0] != 0 and bias_gradient != 0
        assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
print('Trajectory-score checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Stable Bernoulli trajectory log probability and score gradient."""
import numpy as np


def trajectory_score(states, actions, weights, bias):
    """Return (log action-probability product, grad_w, grad_b).

    states [T,D], actions binary [T], weights [D], finite bias. Valid inputs
    assumed; T may be zero. Environment factors are not included. Sum scores
    over time. Compute in floating point, including for integer inputs.
    Stable for extreme logits, including signed infinite logits from overflow
    when the selected actions have representable log probabilities and scores.
    Raise FloatingPointError for NaN logits or nonfinite computed log
    probability or gradients in float64. Do not mutate inputs.
    Evaluate both sigmoid tails independently in float64. Tails below its
    subnormal range underflow to zero before later score-gradient scaling,
    even when an exact tail times a large state would be representable.
    """
    states, actions, weights = [np.asarray(value, float) for value in (states, actions, weights)]
    bias = float(bias)
    with np.errstate(over='ignore', invalid='ignore', under='ignore'):
        logits = states @ weights + bias
        if np.isnan(logits).any():
            raise FloatingPointError('logits exceed the supported numeric range')
        selected_logits = np.where(actions == 1, logits, -logits)
        log_probability = float((-np.logaddexp(0., -selected_logits)).sum())
        p = np.exp(-np.logaddexp(0., -logits))
        q = np.exp(-np.logaddexp(0., logits))
        scores = np.where(actions == 1, q, -p)
        grad_w, grad_b = states.T@scores, float(scores.sum())
    if not np.isfinite(log_probability) or not np.isfinite(grad_w).all() or not np.isfinite(grad_b):
        raise FloatingPointError('trajectory outputs exceed the supported numeric range')
    return log_probability, grad_w, grad_b


In [ ]:
import numpy as np
x = np.array([[1., 2.], [-1., 3.], [2., 0.]])
a, w, b = np.array([1, 0, 1]), np.array([.2, -.4]), .1
lp, gw, gb = trajectory_score(x, a, w, b)
p = 1/(1+np.exp(-(x@w+b)))
assert np.isclose(np.exp(lp), np.prod(np.where(a, p, 1-p)))
eps = 1e-5
for i in range(2):
    d = np.eye(2)[i]*eps
    assert np.isclose(gw[i], (trajectory_score(x,a,w+d,b)[0]-trajectory_score(x,a,w-d,b)[0])/(2*eps))
assert np.isclose(gb, (trajectory_score(x,a,w,b+eps)[0]-trajectory_score(x,a,w,b-eps)[0])/(2*eps))
for states, actions, weights, bias in [
    (np.array([[200]], dtype=np.uint8), np.array([1], dtype=np.uint8), np.array([2], dtype=np.uint8), np.int16(-400)),
    (np.array([[100, 100]], dtype=np.int8), np.array([0], dtype=np.uint8), np.array([2, -1], dtype=np.int8), np.int16(-100)),
    (np.array([[-100]], dtype=np.int8), np.array([1], dtype=np.int8), np.array([2], dtype=np.int8), np.int16(200)),
]:
    before = [value.copy() for value in (states, actions, weights)]
    log_probability, gradient, bias_gradient = trajectory_score(states, actions, weights, bias)
    score = float(actions[0]) - .5
    assert np.isclose(log_probability, -np.log(2.))
    assert np.array_equal(gradient, states.astype(float)[0] * score) and bias_gradient == score, 'Promote integer operands before matmul and signed cancellation'
    assert gradient.dtype.kind == 'f' and isinstance(log_probability, float) and isinstance(bias_gradient, float)
    assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
assert np.isfinite(trajectory_score(np.array([[1000.]]), np.array([0]), np.ones(1), 0)[0])
for state, action in [(1e308, 1), (-1e308, 0)]:
    states, actions, weights = np.array([[state]]), np.array([action]), np.array([2.])
    before = [value.copy() for value in (states, actions, weights)]
    log_probability, gradient, bias_gradient = trajectory_score(states, actions, weights, 0.)
    assert log_probability == 0. and np.array_equal(gradient, [0.]) and bias_gradient == 0., 'Saturated matching actions must not multiply an unchosen infinite log probability by zero'
    assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
for state, action in [(1e308, 0), (-1e308, 1)]:
    log_probability, gradient, bias_gradient = trajectory_score(np.array([[state]]), np.array([action]), np.ones(1), 0.)
    assert log_probability == -1e308 and np.array_equal(gradient, [-1e308])
    assert bias_gradient == (1. if action else -1.), 'Finite extreme logits retain representable opposing-action scores'
for states, actions, weights, bias in [
    (np.array([[1e308]]), np.array([0]), np.array([2.]), 0.),
    (np.array([[-1e308]]), np.array([1]), np.array([2.]), 0.),
    (np.zeros((2, 1)), np.ones(2), np.zeros(1), -1e308),
    (np.full((4, 1), 1e308), np.ones(4), np.zeros(1), 0.),
    (np.array([[0.]]), np.ones(1), np.array([np.inf]), 0.),
]:
    before = [value.copy() for value in (states, actions, weights)]
    try:
        trajectory_score(states, actions, weights, bias)
    except FloatingPointError:
        pass
    else:
        raise AssertionError('Nonrepresentable trajectory outputs and NaN logits must fail explicitly')
    assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
empty = trajectory_score(np.empty((0,2)), np.array([]), w, b)
assert empty[0] == 0 and np.allclose(empty[1], 0) and empty[2] == 0
tail = np.exp(-40.)/(1+np.exp(-40.))
for sign in [1., -1.]:
    for action in [0, 1]:
        states, actions, weights = np.array([[1e18]]), np.array([action]), np.array([sign*4e-17])
        before = [value.copy() for value in (states, actions, weights)]
        log_probability, gradient, bias_gradient = trajectory_score(states, actions, weights, 0.)
        matching = action == int(sign > 0)
        expected_score = (1. if action else -1.)*(tail if matching else 1-tail)
        expected_log_probability = -np.log1p(np.exp(-40.)) if matching else -40.-np.log1p(np.exp(-40.))
        np.testing.assert_allclose(log_probability, expected_log_probability, rtol=1e-14, atol=0.)
        np.testing.assert_allclose(gradient, [1e18*expected_score], rtol=1e-14, atol=0.)
        np.testing.assert_allclose(bias_gradient, expected_score, rtol=1e-14, atol=0.)
        assert log_probability < 0 and gradient[0] != 0 and bias_gradient != 0
        assert all(np.array_equal(value, original) for value, original in zip((states, actions, weights), before))
print('Trajectory-score checks passed.')


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""Exact expected-return gradient versus Monte Carlo and finite differences."""
from itertools import product
import json
import numpy as np
import torch


def gradient_experiment():
    states = torch.tensor([[1., -1.], [.5, 2.]], dtype=torch.float64)
    theta = torch.tensor([.3, -.2], dtype=torch.float64, requires_grad=True)
    labels = torch.tensor([1., 0.], dtype=torch.float64)
    p = torch.sigmoid(states@theta)
    imitation = (p-labels).abs().mean()
    imitation.backward()
    hand = states.T@((1-2*labels)*p.detach()*(1-p.detach()))/2
    assert torch.allclose(theta.grad, hand)
    theta.grad = None

    def expected_return(weight):
        probabilities = torch.sigmoid(states@weight)
        total = torch.zeros((), dtype=torch.float64)
        for actions in product([0, 1], repeat=2):
            action = torch.tensor(actions)
            likelihood = torch.where(action.bool(), probabilities, 1-probabilities).prod()
            reward = float(actions == (1, 0))
            total = total + likelihood*reward
        return total
    objective = expected_return(theta); objective.backward()
    exact = theta.grad.detach().numpy()
    finite = []
    for i in range(2):
        delta = torch.eye(2, dtype=torch.float64)[i]*1e-5
        finite.append(float(((expected_return(theta.detach()+delta)-expected_return(theta.detach()-delta))/(2e-5))))
    assert np.allclose(exact, finite, atol=1e-8)
    rng = np.random.default_rng(8); prob = torch.sigmoid(states@theta.detach()).numpy()
    actions = (rng.random((20000, 2)) < prob).astype(float)
    reward = ((actions[:, 0] == 1) & (actions[:, 1] == 0)).astype(float)
    scores = (actions-prob)@states.numpy()
    estimates = reward[:, None]*scores
    baseline_estimates = (reward-float(objective.detach()))[:, None]*scores
    # A simple compounding-error model: independent local correctness q over H steps.
    rollout = [{'offline_action_accuracy': q, 'horizon': h, 'all_actions_correct_probability': q**h} for q in [.9,.99] for h in [5,50]]
    return dict(exact_gradient=exact.tolist(), finite_difference=finite,
        monte_carlo=estimates.mean(0).tolist(), samples=len(actions),
        raw_variance=estimates.var(0).tolist(), baseline_variance=baseline_estimates.var(0).tolist(),
        imitation_gradient=hand.tolist(), rollout_shift_toy=rollout,
        limitation='The q^H toy assumes independent local errors; real distribution shift can be more complex.')


if __name__ == '__main__':
    print(json.dumps(gradient_experiment(), indent=2))
